# Cross-cohort pilot diagnostics

A read-only audit of the Stage 3B leave-one-cohort-out predictions. Held-out outcomes are used exclusively for evaluation, not for threshold selection, feature selection, or model tuning.

In [1]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ModuleNotFoundError:
    pass

candidates = [Path("/content/drive/MyDrive/CMPS355_Research"), Path.cwd(), *Path.cwd().parents]
ROOT = next((path for path in candidates if (path / "runs" / "pilot").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Cannot locate CMPS355_Research/runs/pilot")

print("Project:", ROOT)

Mounted at /content/drive
Project: /content/drive/MyDrive/CMPS355_Research


In [2]:
import hashlib
import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    roc_auc_score,
)

reports = sorted((ROOT / "runs" / "pilot").glob("*/report.json"))
if not reports:
    raise FileNotFoundError("No Stage 3B report.json was found in runs/pilot")

SOURCE_REPORT = reports[-1]
SOURCE_DIR = SOURCE_REPORT.parent
report = json.loads(SOURCE_REPORT.read_text(encoding="utf-8"))

if report.get("stage") != "03B" or report.get("status") != "PASS":
    raise RuntimeError("Selected Stage 3B report is not marked PASS")
if report.get("dataset_sha256") != "8a83f20035fcac5ae5d8bfe496e4d7491e47400b61ca1080b828c8ad54e91a74":
    raise RuntimeError("Unexpected dataset checksum")

expected = report.get("artifacts_sha256", {})
required = {
    "inner_scores.csv", "outer_metrics.csv", "predictions.csv",
    "selected_genes.csv", "candidate_genes.csv", "preprocessing.csv"
}
if set(expected) != required:
    raise RuntimeError("The Stage 3B report has an unexpected artifact manifest")

for name, declared_hash in expected.items():
    digest = hashlib.sha256((SOURCE_DIR / name).read_bytes()).hexdigest()
    if digest != declared_hash:
        raise RuntimeError(f"Artifact checksum mismatch: {name}")

prediction = pd.read_csv(SOURCE_DIR / "predictions.csv")
metrics = pd.read_csv(SOURCE_DIR / "outer_metrics.csv")
selected = pd.read_csv(SOURCE_DIR / "selected_genes.csv")
inner = pd.read_csv(SOURCE_DIR / "inner_scores.csv")

print("Source run:", SOURCE_DIR.relative_to(ROOT))
print("Verified artifacts:", len(expected))
print("Samples:", report["n_samples"])
print("Methods:", ", ".join(report["methods"]))

Source run: runs/pilot/20261107T041823175243Z
Verified artifacts: 6
Samples: 689
Methods: lasso, elastic_net


## Integrity and metric verification

In [3]:
methods = {"lasso", "elastic_net"}
cohorts = {"d1", "d2", "d3", "d4"}
if set(prediction["method"]) != methods or set(prediction["test_cohort"]) != cohorts:
    raise RuntimeError("Missing model or held-out cohort")
if set(metrics["method"]) != methods or set(metrics["test_cohort"]) != cohorts:
    raise RuntimeError("The outer-metrics table is incomplete")
if len(metrics) != 8 or metrics.duplicated(["test_cohort", "method"]).any():
    raise RuntimeError("There must be exactly eight distinct outer model fits")
if prediction[["ID", "method"]].duplicated().any():
    raise RuntimeError("A patient has duplicate predictions for the same method")
if not set(prediction["true_CD"].unique()) <= {0, 1}:
    raise RuntimeError("Invalid true labels")
if prediction["probability_CD"].isna().any() or not prediction["probability_CD"].between(0, 1).all():
    raise RuntimeError("Invalid predicted probabilities")
if not (prediction["predicted_CD"].to_numpy() == (prediction["probability_CD"].to_numpy() >= 0.5).astype(int)).all():
    raise RuntimeError("Saved binary labels disagree with the specified threshold")
if (prediction["true_label"].eq("CD").astype(int) != prediction["true_CD"]).any():
    raise RuntimeError("Class coding mismatch")

ids_by_method = {}
for method in sorted(methods):
    subset = prediction.loc[prediction["method"].eq(method)]
    if len(subset) != report["n_samples"] or subset["ID"].nunique() != report["n_samples"]:
        raise RuntimeError(f"Incomplete patient coverage: {method}")
    ids_by_method[method] = set(subset["ID"])
if ids_by_method["lasso"] != ids_by_method["elastic_net"]:
    raise RuntimeError("Methods cover different patient sets")
if prediction.groupby("ID")["test_cohort"].nunique().ne(1).any():
    raise RuntimeError("A patient appears in more than one held-out cohort")
if prediction.groupby("ID")["true_CD"].nunique().ne(1).any():
    raise RuntimeError("Inconsistent outcomes across methods")

checks = []
for (cohort, method), subset in prediction.groupby(["test_cohort", "method"], sort=True):
    documented = metrics.loc[metrics["test_cohort"].eq(cohort) & metrics["method"].eq(method)].iloc[0]
    y = subset["true_CD"].to_numpy()
    prob = subset["probability_CD"].to_numpy()
    binary = subset["predicted_CD"].to_numpy()
    tn, fp, fn, tp = confusion_matrix(y, binary, labels=[0, 1]).ravel()
    recalculated = {
        "auroc": roc_auc_score(y, prob),
        "average_precision": average_precision_score(y, prob),
        "balanced_accuracy": balanced_accuracy_score(y, binary),
        "sensitivity": tp / (tp + fn),
        "specificity": tn / (tn + fp),
        "true_positive": tp,
        "false_positive": fp,
        "true_negative": tn,
        "false_negative": fn,
        "n_positive": tp + fn,
        "n_negative": tn + fp,
        "n_test_samples": len(subset),
    }
    for name, value in recalculated.items():
        if not np.isclose(float(documented[name]), float(value), rtol=1e-10, atol=1e-10):
            raise RuntimeError(f"Saved metric mismatch: {cohort} / {method} / {name}")
    n_selected = len(selected.loc[selected["test_cohort"].eq(cohort) & selected["method"].eq(method)])
    if n_selected != int(documented["n_nonzero_coefficients"]):
        raise RuntimeError(f"Selected-gene count mismatch: {cohort} / {method}")
    checks.append({"test_cohort": cohort, "method": method, "audit": "PASS"})

if inner[["test_cohort", "validation_cohort", "method", "C"]].duplicated().any():
    raise RuntimeError("Duplicate inner-fold settings")
if len(inner) != 72 or not inner["converged"].astype(bool).all():
    raise RuntimeError("Incomplete or unconverged inner-fold fits")
for record in metrics.itertuples(index=False):
    subset = inner.loc[inner["test_cohort"].eq(record.test_cohort) & inner["method"].eq(record.method)]
    if subset["validation_cohort"].nunique() != 3:
        raise RuntimeError("Missing inner validation cohort")
    group_means = subset.groupby("C")["auroc"].mean().sort_index()
    best = group_means[group_means >= group_means.max() - 1e-12].index.min()
    if not np.isclose(best, record.C):
        raise RuntimeError("Outer-model C differs from inner cohort selection")
    if not np.isclose(group_means.loc[record.C], record.mean_inner_auroc):
        raise RuntimeError("Inner AUROC does not match recorded model selection")

print("STAGE 3B ARTIFACT AUDIT: PASS")
display(pd.DataFrame(checks))

STAGE 3B ARTIFACT AUDIT: PASS


,test_cohort,method,audit
0,d1,elastic_net,PASS
1,d1,lasso,PASS
2,d2,elastic_net,PASS
3,d2,lasso,PASS
4,d3,elastic_net,PASS
5,d3,lasso,PASS
6,d4,elastic_net,PASS
7,d4,lasso,PASS


## Cohort-level operating characteristics

Balanced accuracy is computed at the prespecified 0.5 threshold. The mean predicted probability, observed prevalence and predicted-positive rate are descriptive diagnostics, not calibration corrections.

In [4]:
rows = []
for (cohort, method), subset in prediction.groupby(["test_cohort", "method"], sort=True):
    base = metrics.loc[metrics["test_cohort"].eq(cohort) & metrics["method"].eq(method)].iloc[0]
    rows.append({
        "cohort": cohort,
        "method": method,
        "n": len(subset),
        "CD_prevalence": subset["true_CD"].mean(),
        "predicted_positive_rate": subset["predicted_CD"].mean(),
        "mean_predicted_probability": subset["probability_CD"].mean(),
        "AUROC": base.auroc,
        "average_precision": base.average_precision,
        "balanced_accuracy": base.balanced_accuracy,
        "sensitivity": base.sensitivity,
        "specificity": base.specificity,
        "Brier_score": brier_score_loss(subset["true_CD"], subset["probability_CD"]),
        "C": base.C,
        "selected_genes": int(base.n_nonzero_coefficients),
    })
cohort_summary = pd.DataFrame(rows)
display(cohort_summary.round(3))

,cohort,method,n,CD_prevalence,predicted_positive_rate,mean_predicted_probability,AUROC,average_precision,balanced_accuracy,sensitivity,specificity,Brier_score,C,selected_genes
0,d1,elastic_net,260,0.838,0.992,0.821,0.866,0.974,0.495,0.991,0.000,0.112,0.03,14
1,d1,lasso,260,0.838,0.996,0.752,0.832,0.966,0.498,0.995,0.000,0.125,0.03,5
2,d2,elastic_net,179,0.749,0.369,0.450,0.655,0.839,0.628,0.433,0.822,0.295,0.03,28
3,d2,lasso,179,0.749,0.799,0.795,0.662,0.846,0.559,0.828,0.289,0.243,3.00,225
4,d3,elastic_net,76,0.447,0.461,0.468,0.734,0.749,0.669,0.647,0.690,0.214,0.30,138
5,d3,lasso,76,0.447,0.434,0.472,0.661,0.617,0.613,0.559,0.667,0.294,3.00,251
6,d4,elastic_net,174,0.626,0.161,0.313,0.785,0.878,0.616,0.248,0.985,0.294,0.03,20
7,d4,lasso,174,0.626,0.138,0.299,0.775,0.868,0.598,0.211,0.985,0.306,0.03,6


## Source BioProject breakdown

BioProject-level summaries are descriptive. The smallest studies have limited sample counts; these summaries must not be used for model selection or post hoc threshold adjustment.

In [5]:
study_rows = []
for (cohort, study, method), subset in prediction.groupby(["test_cohort", "Study", "method"], sort=True):
    y = subset["true_CD"]
    study_rows.append({
        "cohort": cohort,
        "study": study,
        "method": method,
        "n": len(subset),
        "CD": int(y.sum()),
        "Not_IBD": int((1 - y).sum()),
        "AUROC": roc_auc_score(y, subset["probability_CD"]) if y.nunique() == 2 else np.nan,
        "balanced_accuracy": balanced_accuracy_score(y, subset["predicted_CD"]) if y.nunique() == 2 else np.nan,
        "predicted_positive_rate": subset["predicted_CD"].mean(),
    })
study_summary = pd.DataFrame(study_rows)
display(study_summary.round(3))

,cohort,study,method,n,CD,Not_IBD,AUROC,balanced_accuracy,predicted_positive_rate
0,d1,PRJNA248469,elastic_net,260,218,42,0.866,0.495,0.992
1,d1,PRJNA248469,lasso,260,218,42,0.832,0.498,0.996
2,d2,PRJNA565216,elastic_net,149,112,37,0.684,0.634,0.282
3,d2,PRJNA565216,lasso,149,112,37,0.641,0.546,0.799
4,d2,PRJNA985602,elastic_net,30,22,8,0.705,0.619,0.800
5,d2,PRJNA985602,lasso,30,22,8,0.767,0.619,0.800
6,d3,PRJNA702434,elastic_net,76,34,42,0.734,0.669,0.461
7,d3,PRJNA702434,lasso,76,34,42,0.661,0.613,0.434
8,d4,PRJNA728646,elastic_net,22,17,5,0.800,0.647,0.227
9,d4,PRJNA728646,lasso,22,17,5,0.682,0.618,0.182


## Descriptive paired AUROC differences

Percentile intervals use stratified patient-level resampling inside each held-out cohort, with identical resampled patients for both methods. These are exploratory uncertainty summaries and are not evidence of independent cohort replication.

In [6]:
rng = np.random.default_rng(355)
paired = prediction.pivot(index=["ID", "test_cohort", "true_CD"], columns="method", values="probability_CD").reset_index()
interval_rows = []
for cohort, subset in paired.groupby("test_cohort", sort=True):
    y = subset["true_CD"].to_numpy()
    p_lasso = subset["lasso"].to_numpy()
    p_elastic = subset["elastic_net"].to_numpy()
    positive = np.flatnonzero(y == 1)
    negative = np.flatnonzero(y == 0)
    bootstrap = np.empty(500)
    for k in range(len(bootstrap)):
        sample_idx = np.concatenate([
            rng.choice(positive, size=len(positive), replace=True),
            rng.choice(negative, size=len(negative), replace=True),
        ])
        bootstrap[k] = (
            roc_auc_score(y[sample_idx], p_elastic[sample_idx])
            - roc_auc_score(y[sample_idx], p_lasso[sample_idx])
        )
    interval_rows.append({
        "cohort": cohort,
        "AUROC_difference_elastic_minus_lasso": roc_auc_score(y, p_elastic) - roc_auc_score(y, p_lasso),
        "bootstrap_95pct_low": np.percentile(bootstrap, 2.5),
        "bootstrap_95pct_high": np.percentile(bootstrap, 97.5),
        "bootstrap_replicates": len(bootstrap),
    })
paired_auroc = pd.DataFrame(interval_rows)
display(paired_auroc.round(3))

,cohort,AUROC_difference_elastic_minus_lasso,bootstrap_95pct_low,bootstrap_95pct_high,bootstrap_replicates
0,d1,0.035,0.014,0.058,500
1,d2,-0.007,-0.092,0.093,500
2,d3,0.073,0.000,0.145,500
3,d4,0.010,-0.033,0.050,500


## Exploratory feature overlap

LOCO training sets share two cohorts. Their selected-gene overlap must not be interpreted as independent cross-cohort biomarker reproducibility.

In [7]:
from itertools import combinations

overlap_rows = []
for method in sorted(methods):
    sets = {
        cohort: set(gene_rows["gene"])
        for cohort, gene_rows in selected.loc[selected["method"].eq(method)].groupby("test_cohort")
    }
    for a, b in combinations(sorted(sets), 2):
        union = sets[a] | sets[b]
        overlap_rows.append({
            "method": method,
            "held_out_a": a,
            "held_out_b": b,
            "n_a": len(sets[a]),
            "n_b": len(sets[b]),
            "intersection": len(sets[a] & sets[b]),
            "jaccard": len(sets[a] & sets[b]) / len(union) if union else np.nan,
        })
gene_overlap = pd.DataFrame(overlap_rows)
display(gene_overlap.round(3))

,method,held_out_a,held_out_b,n_a,n_b,intersection,jaccard
0,elastic_net,d1,d2,14,28,8,0.235
1,elastic_net,d1,d3,14,138,8,0.056
2,elastic_net,d1,d4,14,20,5,0.172
3,elastic_net,d2,d3,28,138,16,0.107
4,elastic_net,d2,d4,28,20,11,0.297
5,elastic_net,d3,d4,138,20,15,0.105
6,lasso,d1,d2,5,225,4,0.018
7,lasso,d1,d3,5,251,3,0.012
8,lasso,d1,d4,5,6,2,0.222
9,lasso,d2,d3,225,251,153,0.474


## Save audit outputs

In [8]:
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
output = ROOT / "runs" / "pilot_diagnostics" / run_id
output.mkdir(parents=True, exist_ok=False)

artifacts = {
    "cohort_summary.csv": cohort_summary,
    "study_summary.csv": study_summary,
    "paired_auroc.csv": paired_auroc,
    "gene_overlap.csv": gene_overlap,
    "audit_checks.csv": pd.DataFrame(checks),
}
for name, table in artifacts.items():
    table.to_csv(output / name, index=False)

result = {
    "stage": "03C",
    "status": "PASS",
    "source_report": str(SOURCE_REPORT.relative_to(ROOT)),
    "source_run_sha256": hashlib.sha256(SOURCE_REPORT.read_bytes()).hexdigest(),
    "dataset_sha256": report["dataset_sha256"],
    "source_artifact_sha256": expected,
    "methods": sorted(methods),
    "n_unique_patients": int(prediction["ID"].nunique()),
    "n_audited_outer_folds": len(checks),
    "n_inner_validation_evaluations": len(inner),
    "recalculated_outer_metrics": True,
    "threshold_adjustment_performed": False,
    "gene_overlap_role": "exploratory_overlapping_training_sets",
    "software": {"pandas": pd.__version__, "numpy": np.__version__, "scikit_learn": sklearn.__version__},
    "outputs_sha256": {
        name: hashlib.sha256((output / name).read_bytes()).hexdigest()
        for name in artifacts
    },
    "created_utc": datetime.now(timezone.utc).isoformat(),
}
(output / "report.json").write_text(json.dumps(result, indent=2), encoding="utf-8")

print("STAGE 3C DIAGNOSTIC AUDIT: PASS")
print("Saved:", output.relative_to(ROOT))

STAGE 3C DIAGNOSTIC AUDIT: PASS
Saved: runs/pilot_diagnostics/20261107T042708846051Z
